# 실습 5회: RAG · 스트리밍 생성 · Agentic 재무 분석 시스템

서울대학교 핀테크 실습 과정 | 12기

---

## 학습 목표

1. **RAG** 원리를 이해하고 감사보고서 벡터 DB를 구축한다.
2. **스트리밍 생성**으로 LLM 답변을 실시간 토큰 단위로 출력한다.
3. **Agentic 설계**의 핵심인 Tool Calling과 ReAct 루프를 구현한다.
4. RAG + 스트리밍 + Tool을 결합한 **재무 분석 에이전트 챗봇**을 완성한다.

---

## 전체 아키텍처

```
[사용자 질문]
      |
  [Step 3] RAG 검색
  감사보고서 청크 --> 임베딩 --> FAISS 벡터 DB --> 유사 문서 검색
      |
  [Step 4] RAG Q&A 파이프라인 (Qwen3)
      |
  [Step 5] 스트리밍 생성 (TextIteratorStreamer)
      |
  [Step 6] Agentic 설계 (Tool Calling + ReAct)
  도구: 재무 조회 / 비율 계산 / RAG 검색
  루프: Thought -> Action -> Observation -> Answer
      |
  [Step 7] 통합 재무 분석 챗봇
  멀티턴 + 스트리밍 + RAG + Tool
```

---

## 사용 기술 스택

| 레이어 | 라이브러리 | 역할 |
|:---|:---|:---|
| 임베딩 | sentence-transformers | 텍스트 -> 벡터 |
| 벡터 DB | faiss-cpu | 유사도 검색 |
| LLM | Qwen/Qwen3-0.6B | 추론 · 생성 |
| 스트리밍 | TextIteratorStreamer | 실시간 출력 |
| Tool | Python 함수 + JSON 스키마 | Agentic 도구 |


---

## Step 1. 패키지 설치

| 패키지 | 용도 | Colab/CUDA | Mac | CPU |
|:---|:---|:---:|:---:|:---:|
| sentence-transformers | 텍스트 임베딩 | O | O | O |
| faiss-cpu | 벡터 유사도 검색 | O | O | O |
| transformers (>=4.51) | Qwen3 모델/토크나이저 | O | O | O |
| accelerate | 디바이스 설정 | O | O | O |


In [6]:
import sys, subprocess, platform

try:
    import google.colab; _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

_IS_MAC = (platform.system() == "Darwin")

# 공통 패키지
PKGS = [
    "transformers>=4.51.0",
    "sentence-transformers",
    "faiss-cpu",
    "accelerate",
    "datasets",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PKGS, check=True)

# CUDA 전용: bitsandbytes (4-bit 양자화 필수)
# Mac / CPU 환경에서는 설치하지 않아 충돌을 방지한다.
if not _IS_MAC:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.43.0"],
        check=False
    )

print("패키지 설치 완료")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


패키지 설치 완료


In [7]:
import torch, platform

try:
    import google.colab; IN_COLAB = True
except ImportError:
    IN_COLAB = False

IS_MAC = (platform.system() == "Darwin")

if torch.cuda.is_available():
    DEVICE = "cuda"
    TORCH_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif IS_MAC and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    # MPS 는 float16 연산 지원이 불완전 → float32 로 고정한다.
    # float32 는 float16 대비 2배 메모리를 사용하지만 커널 사망을 방지한다.
    TORCH_DTYPE = torch.float32
else:
    DEVICE = "cpu"
    TORCH_DTYPE = torch.float32

ENV_CONFIG = {"in_colab": IN_COLAB, "is_mac": IS_MAC,
              "device": DEVICE, "torch_dtype": TORCH_DTYPE}

env_label = "Google Colab" if IN_COLAB else ("Mac (MPS)" if IS_MAC else "로컬 CUDA")
print(f"환경: {env_label}  |  디바이스: {DEVICE.upper()}  |  dtype: {TORCH_DTYPE}")

if DEVICE == "cuda":
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram_gb:.1f} GB")
    if vram_gb < 8:
        print("경고: VRAM < 8 GB. 4-bit 양자화 필수, max_new_tokens 를 줄이세요.")


환경: Mac (MPS)  |  디바이스: MPS  |  dtype: torch.float16


---

## Step 2. 파일 준비

3회차에서 생성한 출력물을 사용한다.

| 파일 | RAG 용도 |
|:---|:---|
| sections.json | 감사보고서 전체 텍스트 -> 청크 -> 임베딩 |
| 2024_*.csv (5개) | 재무 수치 조회 Tool용 |

- **Colab**: `/content` 에 업로드 또는 Drive 마운트
- **로컬**: `../output/processed/` 경로 자동 사용


In [8]:
import os, json
from pathlib import Path

if ENV_CONFIG["in_colab"]:
    TABLES_DIR    = Path("/content")
    SECTIONS_PATH = Path("/content/sections.json")
    OUTPUT_DIR    = Path("/content/rag_output")
elif ENV_CONFIG["device"] == "mps":
    TABLES_DIR    = Path("../output/processed/tables/2024")
    SECTIONS_PATH = Path("../output/processed/jsons/2024/sections.json")
    OUTPUT_DIR    = Path("../output/rag_output")
else:
    TABLES_DIR    = Path("../output/processed/tables/2024")
    SECTIONS_PATH = Path("../output/processed/jsons/2024/sections.json")
    OUTPUT_DIR    = Path("../output/rag_output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [SECTIONS_PATH,
          TABLES_DIR/"2024_BS_재무상태표.csv",
          TABLES_DIR/"2024_IS_손익계산서.csv",
          TABLES_DIR/"2024_CF_현금흐름표.csv"]:
    print(f'  [{"OK" if p.exists() else "없음"}] {p.name}')


  [OK] sections.json
  [OK] 2024_BS_재무상태표.csv
  [OK] 2024_IS_손익계산서.csv
  [OK] 2024_CF_현금흐름표.csv


---

## Step 3. RAG (Retrieval-Augmented Generation)

### 3.1 RAG란 무엇인가

LLM은 학습 시점 이후의 정보나 특정 도메인 데이터를 알지 못한다.
**RAG**는 질문에 앞서 외부 문서 DB에서 관련 정보를 검색하고,
그 결과를 프롬프트에 포함시켜 LLM이 정확하게 답변하도록 돕는다.

```
[일반 LLM]  질문 --> 모델 파라미터 기억 --> 답변  (환각 위험)

[RAG]  질문 --> 임베딩 변환 --> 벡터 DB 검색 --> 관련 문서 k개
              --> 프롬프트: 문서 + 질문 --> LLM --> 답변  (근거 있는 답변)
```

---

### 3.2 임베딩(Embedding)이란

텍스트를 **의미를 담은 고차원 실수 벡터**로 변환하는 과정이다.
의미가 비슷한 문장은 벡터 공간에서 가까이 위치한다.

```
"매출액은 얼마인가?" --> [0.12, -0.34, 0.87, ...]  (384차원)
"영업이익은 얼마인가?" --> [0.11, -0.31, 0.85, ...]  (가까움)
"날씨가 어때?" --> [-0.73, 0.45, -0.21, ...] (멀리)
```

---

### 3.3 FAISS 벡터 DB

Meta(Facebook AI)가 개발한 고속 벡터 유사도 검색 라이브러리.

| 특징 | 내용 |
|:---|:---|
| 검색 방식 | L2 거리(Euclidean) 또는 코사인 유사도 |
| 확장성 | 수백만 벡터도 밀리초 단위 검색 |
| 인덱스 | IndexFlatL2 (정확) / IndexIVFFlat (근사, 대용량) |

---

### 3.4 청킹(Chunking) 전략

문서를 그대로 임베딩하면 너무 길어 검색 정확도가 떨어진다.

| 전략 | 설명 | 적합한 경우 |
|:---|:---|:---|
| 고정 크기 | 토큰/문자 수 기준 분할 | 균일한 텍스트 |
| 문장 단위 | 마침표 기준 분할 | 짧은 문단 |
| 섹션 단위 | 제목 기준 분할 | **감사보고서** (여기서 사용) |
| 슬라이딩 윈도우 | overlap 있는 분할 | 맥락 보존 필요 |


In [9]:
# [실습 3-1] 임베딩 모델 로드
# paraphrase-multilingual-MiniLM-L12-v2: 50개 언어 지원, 384차원

# 임베딩 모델은 항상 CPU 에서 실행한다.
# LLM 이 GPU/MPS VRAM 을 점유하므로 충돌 방지를 위해 device="cpu" 를 명시한다.
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "snunlp/KR-SBERT-V40K-klueNLI-augSTS"
# sentence-transformers 는 MPS 지원이 불안정하므로 CPU 사용
embed_device = "cpu" if DEVICE == "mps" else DEVICE
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=embed_device)

print(f"임베딩 모델: {EMBED_MODEL_NAME}")
print(f"임베딩 차원: {embed_model.get_sentence_embedding_dimension()}")
print(f"실행 디바이스: {embed_device}")

test_sentences = [
    "삼성전자의 매출액은 얼마인가?",
    "영업이익은 얼마인가?",
    "날씨가 어떻습니까?",
]
test_vecs = embed_model.encode(test_sentences, normalize_embeddings=True)
sim_01 = (test_vecs[0] * test_vecs[1]).sum()
sim_02 = (test_vecs[0] * test_vecs[2]).sum()
print(f"\n[코사인 유사도]")
print(f"  매출액 vs 영업이익: {sim_01:.4f}  (의미 유사 -> 높아야 함)")
print(f"  매출액 vs 날씨:    {sim_02:.4f}  (의미 무관 -> 낮아야 함)")


임베딩 모델: snunlp/KR-SBERT-V40K-klueNLI-augSTS
임베딩 차원: 768
실행 디바이스: cpu

[코사인 유사도]
  매출액 vs 영업이익: 0.7803  (의미 유사 -> 높아야 함)
  매출액 vs 날씨:    0.2120  (의미 무관 -> 낮아야 함)


In [10]:
# [실습 3-2] 감사보고서 + 재무 CSV -> 청크 문서 리스트 생성

import re
import pandas as pd

CHUNK_SIZE = 400
OVERLAP    = 50

def split_into_chunks(text, source, max_chars=CHUNK_SIZE):
    text = re.sub(r"[\s\xa0]{3,}", " ", text).strip()
    if not text:
        return []
    if len(text) <= max_chars:
        return [{"text": text, "source": source}]

    # 이중 개행으로 먼저 단락 분리
    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]

    chunks, buf = [], ""
    for para in paragraphs:
        # 핵심 수정: 단일 단락이 max_chars 를 초과하면 글자 단위로 강제 분할
        # 이 분기가 없으면 감사보고서의 장문 단락이 청크 1개로 통째로 들어가
        # 수만 토큰 프롬프트 → CUDA OOM 을 유발한다.
        if len(para) > max_chars:
            if buf.strip():
                chunks.append({"text": buf.strip(), "source": source})
                buf = ""
            step = max_chars - OVERLAP
            for i in range(0, len(para), step):
                piece = para[i:i + max_chars].strip()
                if piece:
                    chunks.append({"text": piece, "source": source})
            continue

        if len(buf) + len(para) > max_chars and buf:
            chunks.append({"text": buf.strip(), "source": source})
            buf = buf[-OVERLAP:] + " " + para
        else:
            buf = (buf + " " + para).strip()

    if buf.strip():
        chunks.append({"text": buf.strip(), "source": source})
    return chunks

all_chunks = []

if SECTIONS_PATH.exists():
    with open(SECTIONS_PATH, encoding="utf-8") as f:
        sections_data = json.load(f)
    for sec in sections_data.get("sections", []):
        sid  = sec.get("section_id", "")
        name = sec.get("section_name", sid)
        text = sec.get("content", "") or sec.get("text", "")
        all_chunks += split_into_chunks(text, source=f"sections/{name}")
    print(f"sections.json -> {len(all_chunks)}개 청크")
else:
    sample = [
        {"text": "삼정회계법인은 삼성전자의 2024년 재무제표에 대해 적정의견을 표명하였습니다."
                 " K-IFRS 기준으로 작성되었으며 중요성 관점에서 공정하게 표시되어 있습니다.",
         "source": "fallback/감사의견"},
        {"text": "삼성전자 2024년 매출액은 209,052,241 백만원으로 전기 대비 22.7% 증가하였습니다."
                 " 영업이익은 12,361,034 백만원으로 흑자전환하였습니다.",
         "source": "fallback/실적요약"},
    ]
    all_chunks = sample
    print(f"sections.json 없음 -- 샘플 텍스트 {len(all_chunks)}개 사용")

for fname, tag in [
    ("2024_BS_재무상태표.csv", "BS"), ("2024_IS_손익계산서.csv", "IS"),
    ("2024_CF_현금흐름표.csv", "CF"), ("2024_CSE_자본변동표.csv", "CSE"),
]:
    p = TABLES_DIR / fname
    if not p.exists(): continue
    df = pd.read_csv(p)
    rows = []
    for _, row in df.iterrows():
        line = " | ".join(str(v) for v in row.values if str(v).strip() and str(v) != "nan")
        if line: rows.append(line)
    all_chunks += split_into_chunks("\n".join(rows), source=f"csv/{tag}")

print(f"최종 청크: {len(all_chunks)}개")
# 청크 크기 통계 출력 (비정상 청크 조기 탐지)
if all_chunks:
    lens = [len(c['text']) for c in all_chunks]
    print(f"청크 길이 — 최소: {min(lens)}, 최대: {max(lens)}, 평균: {sum(lens)//len(lens)}자")
    oversized = [c for c in all_chunks if len(c['text']) > CHUNK_SIZE * 2]
    if oversized:
        print(f"경고: {len(oversized)}개 청크가 CHUNK_SIZE*2({CHUNK_SIZE*2}자)를 초과합니다.")
for c in all_chunks[:2]:
    print(f"  출처: {c['source']}")
    print(f"  텍스트: {c['text'][:100]}...")


sections.json -> 5개 청크
최종 청크: 9개
  출처: sections/독립된 감사인의 감사보고서
  텍스트: 독립된 감사인의 감사보고서 감사의견 우리는 별첨된 삼성전자주식회사(이하 "회사")의 재무제표를 감사하였습니다. 해당 재무제표는 2024년 12월 31일과 2023년 12월 31일 ...
  출처: sections/(첨부)재 무 제 표
  텍스트: (첨부)재 무 제 표 주석 1. 일반적 사항: 삼성전자주식회사(이하 "회사")는 1969년 대한민국에서 설립되어 1975년에 대한민국의 증권거래소에 상장하였습니다. 회사의 사업은 ...


In [11]:
# [실습 3-3] FAISS 벡터 DB 구축

import faiss
import numpy as np

print(f"임베딩 생성 중... ({len(all_chunks)}개 청크)")
texts = [c["text"] for c in all_chunks]
vectors = embed_model.encode(texts, batch_size=32, show_progress_bar=True,
                              normalize_embeddings=True)
vectors = vectors.astype(np.float32)

DIM = vectors.shape[1]
faiss_index = faiss.IndexFlatIP(DIM)  # Inner Product = 코사인 유사도 (정규화 후)
faiss_index.add(vectors)

print(f"FAISS 인덱스 구축 완료")
print(f"  벡터 차원: {DIM}")
print(f"  인덱스 크기: {faiss_index.ntotal}개")

faiss.write_index(faiss_index, str(OUTPUT_DIR / "audit_index.faiss"))
with open(OUTPUT_DIR / "audit_chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=1)
print(f"저장 완료: {OUTPUT_DIR}")


임베딩 생성 중... (9개 청크)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS 인덱스 구축 완료
  벡터 차원: 768
  인덱스 크기: 9개
저장 완료: ../output/rag_output


In [12]:
# [실습 3-4] 유사도 검색 함수 및 테스트

def retrieve(query, k=4):
    """질문과 가장 유사한 청크 k개를 반환한다."""
    q_vec = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = faiss_index.search(q_vec, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1: continue
        results.append({"score": float(score),
                         "text":  all_chunks[idx]["text"],
                         "source": all_chunks[idx]["source"]})
    return results

test_queries = [
    "감사 의견은 무엇인가요?",
    "2024년 매출액과 영업이익",
    "현금흐름 변화",
]
print("=" * 65)
for q in test_queries:
    results = retrieve(q, k=2)
    print(f"Q: {q}")
    for i, r in enumerate(results):
        print(f"  [{i+1}] score={r['score']:.4f} | 출처: {r['source']}")
        print(f"       {r['text'][:100]}...")
    print()


Q: 감사 의견은 무엇인가요?
  [1] score=0.3799 | 출처: sections/외부감사 실시내용
       외부감사 실시내용 1. 감사대상업무 2. 감사참여자 구분별 인원수 및 감사시간 3. 주요 감사실시내용 4. 감사(감사위원회)와의 커뮤니케이션...
  [2] score=0.3244 | 출처: sections/독립된 감사인의 감사보고서
       독립된 감사인의 감사보고서 감사의견 우리는 별첨된 삼성전자주식회사(이하 "회사")의 재무제표를 감사하였습니다. 해당 재무제표는 2024년 12월 31일과 2023년 12월 31일 ...

Q: 2024년 매출액과 영업이익
  [1] score=0.4719 | 출처: sections/독립된 감사인의 감사보고서
       독립된 감사인의 감사보고서 감사의견 우리는 별첨된 삼성전자주식회사(이하 "회사")의 재무제표를 감사하였습니다. 해당 재무제표는 2024년 12월 31일과 2023년 12월 31일 ...
  [2] score=0.4160 | 출처: sections/(첨부)재 무 제 표
       (첨부)재 무 제 표 주석 1. 일반적 사항: 삼성전자주식회사(이하 "회사")는 1969년 대한민국에서 설립되어 1975년에 대한민국의 증권거래소에 상장하였습니다. 회사의 사업은 ...

Q: 현금흐름 변화
  [1] score=0.3300 | 출처: sections/독립된 감사인의 감사보고서
       독립된 감사인의 감사보고서 감사의견 우리는 별첨된 삼성전자주식회사(이하 "회사")의 재무제표를 감사하였습니다. 해당 재무제표는 2024년 12월 31일과 2023년 12월 31일 ...
  [2] score=0.3236 | 출처: sections/주석
       주석 1. 일반적 사항: 삼성전자주식회사(이하 "회사")는 1969년 대한민국에서 설립되어 1975년에 대한민국의 증권거래소에 상장하였습니다. 회사의 사업은 DX 부문, DS 부문...



---

## Step 4. Qwen3 모델 로드 및 RAG Q&A 파이프라인

### 4.1 RAG 프롬프트 설계

RAG에서 프롬프트는 세 부분으로 구성된다.

```
[SYSTEM]
당신은 삼성전자 감사보고서 전문가입니다.
아래 참고 문서를 바탕으로 답변하세요. 문서에 없는 내용은 모른다고 하세요.

[참고 문서]
1. (검색 결과 1)
2. (검색 결과 2)

[USER]
질문 내용
```

핵심: **"문서에 없으면 모른다"** 라는 지시 -> 환각(Hallucination) 방지


In [13]:
# [실습 4-1] Qwen3 모델 로드 (환경별 분기)
#
# ■ CUDA (Colab T4/A100): 4-bit NF4 양자화 → VRAM ~300 MB
# ■ Mac MPS             : float32 CPU 로드 후 MPS 이동
#                         (float16 은 MPS 에서 커널 사망 유발)
# ■ CPU                 : float32

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen3-0.6B"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def _decide_device():
    if DEVICE == "cuda":
        return "cuda"
    if DEVICE == "mps":
        try:
            import psutil
            avail_gb = psutil.virtual_memory().available / 1e9
        except ImportError:
            avail_gb = 8.0
        print(f"가용 통합 메모리: {avail_gb:.1f} GB")
        # float32 Qwen3-0.6B ≈ 2.4 GB, 여유분 포함 5 GB 이상 확인
        if avail_gb >= 5.0:
            return "mps"
        else:
            print("메모리 부족 → CPU fallback (float32)")
            return "cpu"
    return "cpu"

LOAD_DEVICE = _decide_device()
print(f"모델 로드 디바이스: {LOAD_DEVICE}")

if LOAD_DEVICE == "cuda":
    try:
        from transformers import BitsAndBytesConfig
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=TORCH_DTYPE,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb,
            device_map="auto",
            torch_dtype=TORCH_DTYPE,
            low_cpu_mem_usage=True,
        )
        print("[CUDA] 4-bit NF4 양자화 로드 완료")
    except Exception as e:
        # bitsandbytes 미설치 또는 버전 불일치 시 bfloat16 로 재시도
        print(f"[CUDA] 4-bit 로드 실패 ({e}) → bfloat16 로 재시도")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            device_map="auto",
            torch_dtype=TORCH_DTYPE,
            low_cpu_mem_usage=True,
        )
        print("[CUDA] bfloat16 로드 완료")

elif LOAD_DEVICE == "mps":
    # float32 필수: float16 은 MPS 에서 일부 연산 미지원으로 커널 사망 유발
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
    )
    model = model.to("mps")
    print("[Mac MPS] float32 로드 완료")

else:  # CPU
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
    )
    print("[CPU] float32 로드 완료 (추론 속도 느림)")

model.eval()
ENV_CONFIG["load_device"] = LOAD_DEVICE

total_p = sum(p.numel() for p in model.parameters())
print(f"파라미터: {total_p/1e6:.1f}M")
print(f"dtype   : {next(model.parameters()).dtype}")
print(f"device  : {next(model.parameters()).device}")

if LOAD_DEVICE == "cuda":
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM 사용: {used:.2f} / {total:.1f} GB")


가용 통합 메모리: 239.6 GB
모델 로드 디바이스: mps


`torch_dtype` is deprecated! Use `dtype` instead!


[Mac MPS] float16 로드 완료
파라미터: 596.0M
dtype   : torch.float16
device  : mps:0


In [14]:
# [실습 4-2] RAG Q&A 함수 (Non-streaming)

RAG_SYSTEM = (
    "당신은 삼성전자 2024년 감사보고서 및 재무제표 전문가입니다. "
    "반드시 아래 [참고 문서] 내용을 근거로 답변하세요. "
    "참고 문서에 없는 내용은 \"제공된 자료에서 확인할 수 없습니다\"라고 답하세요."
)

def build_rag_prompt(query, k=3):
    results = retrieve(query, k=k)
    ctx_parts = [f"[문서 {i+1}] (출처: {r['source']}, 유사도: {r['score']:.3f})\n{r['text']}"
                 for i, r in enumerate(results)]
    context = "\n\n".join(ctx_parts)
    system_with_ctx = f"{RAG_SYSTEM}\n\n[참고 문서]\n{context}"
    messages = [
        {"role": "system", "content": system_with_ctx},
        {"role": "user",   "content": query},
    ]
    return messages, results

def _get_device():
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device("cpu")

def _clear_cache():
    # 생성 완료 후 GPU/MPS 캐시를 해제해 메모리 누적을 방지한다.
    dev = _get_device()
    if dev.type == "cuda":
        torch.cuda.empty_cache()
    elif dev.type == "mps":
        torch.mps.empty_cache()

def rag_query(query, thinking=False, max_new_tokens=256):
    messages, results = build_rag_prompt(query)
    text = tokenizer.apply_chat_template(messages, tokenize=False,
               add_generation_prompt=True, enable_thinking=thinking)
    inputs = tokenizer([text], return_tensors="pt").to(_get_device())
    # 입력 토큰 수 확인: 초과 시 잘라내어 CUDA OOM 방지
    n_tokens = inputs["input_ids"].shape[1]
    MAX_INPUT = 3072
    if n_tokens > MAX_INPUT:
        print(f"경고: 입력 토큰 {n_tokens} > {MAX_INPUT} → 자름")
        inputs = tokenizer([text], return_tensors="pt",
                           truncation=True, max_length=MAX_INPUT).to(_get_device())
    gen_kw = dict(max_new_tokens=max_new_tokens, do_sample=True,
                  temperature=0.6 if thinking else 0.7,
                  top_p=0.95 if thinking else 0.8, top_k=20,
                  pad_token_id=tokenizer.eos_token_id)
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kw)
    ids = out[0][len(inputs.input_ids[0]):].tolist()
    try:
        idx = len(ids) - ids[::-1].index(151668)  # </think>
        answer = tokenizer.decode(ids[idx:], skip_special_tokens=True).strip()
    except ValueError:
        answer = tokenizer.decode(ids, skip_special_tokens=True).strip()
    _clear_cache()
    return {"answer": answer, "sources": [r["source"] for r in results]}

print("RAG 함수 준비 완료")
print(f"모델 디바이스: {_get_device()}")


RAG 함수 준비 완료
모델 디바이스: mps:0


In [ ]:
# [실습 4-3] RAG Q&A 테스트

test_questions = [
    "2024년 삼성전자 감사 의견은 무엇입니까?",
    "2024년 매출액은 얼마이고 전기 대비 어떻게 변했습니까?",
    "감사인의 핵심감사사항은 무엇입니까?",
]

for q in test_questions:
    print("=" * 65)
    print(f"Q: {q}")
    r = rag_query(q, thinking=False, max_new_tokens=256)
    print(f"A: {r['answer']}")
    print(f"   [출처: {r['sources']}]")
    print()


---

## Step 5. 스트리밍 생성 (Streaming Generation)

### 5.1 스트리밍이 필요한 이유

일반 `model.generate()`는 전체 답변 완성 후 한 번에 반환한다.
스트리밍은 토큰이 생성될 때마다 즉시 전달한다.

```
[일반]    (5초 대기) --> "삼성전자의 2024년 매출액은 209조원입니다."
[스트리밍] "삼성" --> "전자" --> "의" --> "2024" --> "년" --> ...
```

### 5.2 TextIteratorStreamer 동작 원리

```
메인 스레드                  | 생성 스레드
-----------------------------|---------------------------
streamer = TextIterator(...) |
Thread(generate).start()     | model.generate(streamer=s)
for token in streamer:       |  --> 토큰 생성시 큐에 추가
    print(token, end="")     |     ^ 메인 스레드가 소비
```

`threading.Thread`로 생성을 별도 스레드에서 실행하고,
`TextIteratorStreamer`가 토큰 큐 역할을 한다.


In [ ]:
# [실습 5-1] TextIteratorStreamer 기본 실습

import threading
from transformers import TextIteratorStreamer

def stream_generate(messages, thinking=False, max_new_tokens=256):
    # 메시지를 받아 스트리밍 방식으로 토큰을 제공하는 (streamer, thread) 튜플을 반환한다.
    text = tokenizer.apply_chat_template(messages, tokenize=False,
               add_generation_prompt=True, enable_thinking=thinking)
    inputs = tokenizer([text], return_tensors="pt").to(_get_device())
    n_tokens = inputs["input_ids"].shape[1]
    MAX_INPUT = 3072
    if n_tokens > MAX_INPUT:
        print(f"경고: 입력 토큰 {n_tokens} > {MAX_INPUT} → 자름")
        inputs = tokenizer([text], return_tensors="pt",
                           truncation=True, max_length=MAX_INPUT).to(_get_device())

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
        timeout=120.0,
    )
    gen_kw = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.6 if thinking else 0.7,
        top_p=0.95 if thinking else 0.8,
        top_k=20,
        pad_token_id=tokenizer.eos_token_id,
    )
    # daemon=True: 메인 스레드 종료 시 자동 정리
    thread = threading.Thread(target=model.generate, kwargs=gen_kw, daemon=True)
    thread.start()
    return streamer, thread

msgs = [
    {"role": "system", "content": "당신은 재무 전문가입니다."},
    {"role": "user",   "content": "영업이익률의 의미를 2문장으로 설명해주세요."},
]
print("스트리밍 출력:")
print("-" * 50)
streamer, thread = stream_generate(msgs, thinking=False, max_new_tokens=150)
for token in streamer:
    print(token, end="", flush=True)
thread.join()   # 생성 스레드가 완전히 끝날 때까지 대기
_clear_cache()
print("\n" + "-" * 50)


In [ ]:
# [실습 5-2] 스트리밍 RAG Q&A

def stream_rag_query(query, thinking=False, max_new_tokens=256):
    # RAG 검색 결과를 컨텍스트로 포함한 스트리밍 생성.
    messages, results = build_rag_prompt(query)
    print(f"[검색 문서 {len(results)}개]")
    for r in results:
        print(f"  - {r['source']} (유사도: {r['score']:.3f})")
    print()
    print("[생성 중...]")
    print("-" * 50)
    streamer, thread = stream_generate(messages, thinking=thinking,
                                       max_new_tokens=max_new_tokens)
    full_text = ""
    for token in streamer:
        print(token, end="", flush=True)
        full_text += token
    thread.join()
    _clear_cache()
    print("\n" + "-" * 50)
    return full_text

q = "삼성전자의 2024년 재무 성과를 요약해주세요."
print(f"Q: {q}\n")
_ = stream_rag_query(q, thinking=False, max_new_tokens=300)


---

## Step 6. Agentic 설계 -- Tool Calling + ReAct

### 6.1 Agentic AI란

단순 Q&A를 넘어 **도구를 사용하고 계획을 세워 목표를 달성**하는 AI.

```
[일반 LLM]    질문 --> 답변  (1회 추론)

[Agentic LLM] 질문
               |
             [Thought]  어떤 도구가 필요한가?
               |
             [Action]   도구 호출: get_financial_value("IS", "매출액")
               |
             [Observation]  결과: 209,052,241
               |
             [Thought]  충분한 정보 수집됨. 답변 생성
               |
             [Answer]   최종 답변
```

---

### 6.2 ReAct (Reasoning + Acting)

Yao et al. (2022) 논문에서 제안한 에이전트 프레임워크.

| 단계 | 역할 |
|:---:|:---|
| **Reasoning** | 현재 상황 분석, 다음 행동 결정 |
| **Acting** | 도구 호출 또는 최종 답변 생성 |

---

### 6.3 이 실습에서 사용하는 Tool 4개

| Tool 이름 | 입력 | 용도 |
|:---|:---|:---|
| get_financial_value | statement, item | 재무제표 수치 조회 |
| calculate_ratio | numerator, denominator | 비율(%) 계산 |
| get_yoy_change | statement, item | 전기 대비 증감률 |
| search_audit_report | query | 감사보고서 RAG 검색 |


In [ ]:
# [실습 6-1] 재무 Tool 함수 구현

import pandas as pd

FS_DATA = {}
for fname, tag in [
    ("2024_BS_재무상태표.csv", "BS"), ("2024_IS_손익계산서.csv", "IS"),
    ("2024_CF_현금흐름표.csv", "CF"), ("2024_CSE_자본변동표.csv", "CSE"),
]:
    p = TABLES_DIR / fname
    if p.exists():
        df = pd.read_csv(p)
        df.columns = [str(c).strip() for c in df.columns]
        FS_DATA[tag] = df

def _get_val(df, kw, col_offset=1):
    mask = (df.iloc[:,0].astype(str)
              .str.replace(r"[\s\xa0]+", "", regex=True)
              .str.contains(kw.replace(" ", ""), regex=False))
    rows = df[mask]
    if rows.empty: return None
    for c in df.columns[col_offset:]:
        try:
            v = float(rows.iloc[0][c])
            if not pd.isna(v): return v
        except: pass
    return None

def get_financial_value(statement, item):
    """재무제표(BS/IS/CF/CSE)에서 특정 계정과목의 당기 수치를 조회한다."""
    tag = statement.upper()
    if tag not in FS_DATA:
        return f"오류: {statement} 데이터 없음. 사용 가능: BS, IS, CF, CSE"
    v = _get_val(FS_DATA[tag], item)
    if v is None:
        return f"{statement}에서 \"{item}\" 항목을 찾을 수 없습니다."
    if abs(v) >= 1_000_000:
        return f"{v:,.0f} 백만원 (약 {abs(v)/1e6:.1f}조원)"
    return f"{v:,.0f} 백만원"

def calculate_ratio(numerator_str, denominator_str, label=""):
    """두 수 또는 계정과목명의 비율(%)을 계산한다."""
    def parse(s):
        s = str(s).replace(",", "").replace("백만원", "").strip()
        try: return float(s)
        except:
            for df in FS_DATA.values():
                v = _get_val(df, s)
                if v is not None: return v
            return None
    n, d = parse(numerator_str), parse(denominator_str)
    if n is None: return f"분자 \"{numerator_str}\" 를 찾을 수 없습니다."
    if d is None or d == 0: return "분모가 0이거나 찾을 수 없습니다."
    name = f"{label} " if label else ""
    return f"{name}{n/d*100:.2f}% ({n:,.0f} / {d:,.0f})"

def search_audit_report(query, k=3):
    """감사보고서 벡터 DB에서 관련 내용을 검색한다."""
    results = retrieve(query, k=k)
    if not results: return "관련 내용을 찾지 못했습니다."
    return "\n\n".join(
        f"[{i+1}] (출처: {r['source']})\n{r['text']}"
        for i, r in enumerate(results))

def get_yoy_change(statement, item):
    """특정 계정과목의 당기/전기 수치와 YoY 증감률을 반환한다."""
    tag = statement.upper()
    if tag not in FS_DATA: return f"오류: {statement} 없음"
    df = FS_DATA[tag]
    curr = _get_val(df, item, col_offset=1)
    prev = _get_val(df, item, col_offset=3)
    if curr is None: return f"\"{item}\" 항목을 찾을 수 없습니다."
    if prev is None or prev == 0:
        return f"당기: {curr:,.0f} 백만원 (전기 데이터 없음)"
    rate = (curr - prev) / abs(prev) * 100
    return (f"당기: {curr:,.0f} | 전기: {prev:,.0f} | "
            f"YoY {abs(rate):.1f}% {'증가' if rate >= 0 else '감소'}")

print("[Tool 테스트]")
print(" get_financial_value(IS, 매출액) ->", get_financial_value("IS", "매출액"))
print(" calculate_ratio(영업이익, 매출액, 영업이익률) ->",
      calculate_ratio("영업이익", "매출액", "영업이익률"))
print(" get_yoy_change(BS, 자산총계) ->", get_yoy_change("BS", "자산총계"))


In [ ]:
# [실습 6-2] Tool 레지스트리 및 JSON 스키마

TOOLS = [
    {"type": "function", "function": {
        "name": "get_financial_value",
        "description": "재무제표(BS=재무상태표, IS=손익계산서, CF=현금흐름표, CSE=자본변동표)에서"
                       " 특정 계정과목의 당기 수치를 조회한다.",
        "parameters": {"type": "object",
            "properties": {
                "statement": {"type": "string",
                              "description": "재무제표 종류. BS, IS, CF, CSE 중 하나."},
                "item":      {"type": "string",
                              "description": "조회할 계정과목명 (예: 매출액, 자산총계)"},
            }, "required": ["statement", "item"]},
    }},
    {"type": "function", "function": {
        "name": "calculate_ratio",
        "description": "두 계정과목 또는 수치의 비율(%)을 계산한다.",
        "parameters": {"type": "object",
            "properties": {
                "numerator_str":   {"type": "string", "description": "분자 (계정과목명 또는 수치)"},
                "denominator_str": {"type": "string", "description": "분모"},
                "label":           {"type": "string", "description": "비율 이름 (예: 영업이익률)"},
            }, "required": ["numerator_str", "denominator_str"]},
    }},
    {"type": "function", "function": {
        "name": "search_audit_report",
        "description": "감사보고서 원문에서 질문과 관련된 내용을 검색한다.",
        "parameters": {"type": "object",
            "properties": {
                "query": {"type": "string", "description": "검색할 내용"},
                "k":     {"type": "integer", "description": "반환할 결과 수 (기본 3)"},
            }, "required": ["query"]},
    }},
    {"type": "function", "function": {
        "name": "get_yoy_change",
        "description": "특정 계정과목의 당기/전기 수치와 전기 대비 증감률을 반환한다.",
        "parameters": {"type": "object",
            "properties": {
                "statement": {"type": "string", "description": "BS, IS, CF, CSE 중 하나"},
                "item":      {"type": "string", "description": "계정과목명"},
            }, "required": ["statement", "item"]},
    }},
]

TOOL_FUNCS = {
    "get_financial_value":  get_financial_value,
    "calculate_ratio":      calculate_ratio,
    "search_audit_report":  search_audit_report,
    "get_yoy_change":       get_yoy_change,
}

print(f"등록된 Tool: {list(TOOL_FUNCS.keys())}")


In [ ]:
# [실습 6-3] ReAct 에이전트 루프

AGENT_SYSTEM = (
    "당신은 삼성전자 2024년 감사보고서 전문가 에이전트입니다. "
    "사용자 질문에 답하기 위해 제공된 도구를 적극 활용하세요. "
    "수치 조회는 반드시 도구를 사용하고, 임의로 수치를 생성하지 마세요."
)

def run_agent(user_query, max_turns=5, verbose=True):
    """ReAct 루프: 도구 호출이 없을 때까지 max_turns 반복."""
    history = [
        {"role": "system", "content": AGENT_SYSTEM},
        {"role": "user",   "content": user_query},
    ]
    if verbose:
        print(f"[에이전트] Q: {user_query}")
        print("=" * 65)

    for turn in range(max_turns):
        text = tokenizer.apply_chat_template(
            history, tools=TOOLS, tokenize=False,
            add_generation_prompt=True, enable_thinking=False)
        inputs = tokenizer([text], return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=True,
                                  temperature=0.7, top_p=0.8, top_k=20,
                                  pad_token_id=tokenizer.eos_token_id)
        resp_ids  = out[0][len(inputs.input_ids[0]):].tolist()
        resp_text = tokenizer.decode(resp_ids, skip_special_tokens=False).strip()

        import re as _re
        tool_matches = _re.findall(r"<tool_call>(.*?)</tool_call>", resp_text, _re.DOTALL)

        if not tool_matches:
            clean = _re.sub(r"<[^>]+>", "", resp_text).strip()
            if verbose: print(f"[최종 답변]\n{clean}")
            _clear_cache()
            return clean

        tool_results = []
        for tc_str in tool_matches:
            try:
                tc   = json.loads(tc_str.strip())
                fn   = tc.get("name", "")
                args = tc.get("arguments", {})
                if verbose:
                    print(f"  [Turn {turn+1}] Tool: {fn}({args})")
                res = TOOL_FUNCS[fn](**args) if fn in TOOL_FUNCS else f"알 수 없는 도구: {fn}"
                if verbose: print(f"             결과: {res}")
                tool_results.append({"name": fn, "result": res})
            except json.JSONDecodeError:
                tool_results.append({"name": "error", "result": "JSON 파싱 실패"})

        history.append({"role": "assistant", "content": resp_text})
        for tr in tool_results:
            history.append({"role": "tool", "name": tr["name"],
                            "content": tr["result"]})

    _clear_cache()
    return "(최대 반복 횟수 초과)"

print("ReAct 에이전트 준비 완료")


In [ ]:
# [실습 6-4] 에이전트 테스트 -- 복합 질문

agent_questions = [
    "2024년 영업이익률과 순이익률을 계산하고 비교해주세요.",
    "2024년 부채비율과 유동비율을 구하고 재무 건전성을 평가해주세요.",
    "감사 의견과 핵심 감사사항을 찾아 요약해주세요.",
]

for q in agent_questions:
    print()
    _ = run_agent(q, max_turns=5, verbose=True)
    print("\n" + "=" * 65)


---

## Step 7. 통합 재무 분석 챗봇

RAG + 스트리밍 + Tool Calling을 결합한 **멀티턴 재무 분석 챗봇**을 구현한다.

### 아키텍처

```
[사용자 입력]
      |
  1. RAG 검색 (embed_model + FAISS)
      |
  2. 히스토리 + 검색 결과 --> 프롬프트 구성
      |
  3. Qwen3 생성 (Tool Calling 또는 직접 답변)
      |
  4. Tool 호출 시: 실행 --> 결과 추가 --> 재생성
      |
  5. 스트리밍 출력
```


In [ ]:
# [실습 7-1] 통합 챗봇 클래스

class FinancialChatbot:
    """RAG + Streaming + Tool Calling 통합 재무 분석 챗봇."""

    SYSTEM = (
        "당신은 삼성전자 2024년 감사보고서 전문가 AI입니다. "
        "수치 조회가 필요하면 반드시 도구를 사용하세요. "
        "임의로 수치를 생성하지 마세요. 친절하고 전문적으로 답변하세요."
    )

    def __init__(self):
        self.history = []
        self.turn = 0

    def _build_messages(self, query, rag_ctx=""):
        system = self.SYSTEM
        if rag_ctx:
            system += f"\n\n[관련 문서]\n{rag_ctx}"
        msgs = [{"role": "system", "content": system}]
        msgs += self.history
        msgs.append({"role": "user", "content": query})
        return msgs

    def chat(self, query, use_rag=True, streaming=True, verbose=True):
        self.turn += 1
        if verbose: print(f"\n[Turn {self.turn}] User: {query}")

        # 1. RAG 검색
        rag_ctx = ""
        if use_rag:
            results = retrieve(query, k=3)
            rag_ctx = "\n\n".join(
                f"[문서{i+1}] {r['source']}\n{r['text']}"
                for i, r in enumerate(results))

        messages = self._build_messages(query, rag_ctx)

        # 2. 생성 (Tool Calling 루프)
        response_text = ""
        for _ in range(3):
            text = tokenizer.apply_chat_template(
                messages, tools=TOOLS, tokenize=False,
                add_generation_prompt=True, enable_thinking=False)
            inputs = tokenizer([text], return_tensors="pt").to(model.device)

            if streaming:
                streamer = TextIteratorStreamer(tokenizer, skip_prompt=True,
                                               skip_special_tokens=False, timeout=60.0)
                gen_kw = dict(**inputs, streamer=streamer, max_new_tokens=256,
                              do_sample=True, temperature=0.7, top_p=0.8, top_k=20,
                              pad_token_id=tokenizer.eos_token_id)
                t = threading.Thread(target=model.generate, kwargs=gen_kw)
                t.start()
                response_text = ""
                if verbose: print("Assistant: ", end="", flush=True)
                for tok in streamer:
                    response_text += tok
                    import re as _re
                    clean = tok.replace("<|im_end|>", "").replace("<tool_call>", "")\
                               .replace("</tool_call>", "")
                    if verbose and not _re.match(r"<[^>]+>", clean) and clean.strip():
                        print(clean, end="", flush=True)
                if verbose: print()
            else:
                with torch.no_grad():
                    out = model.generate(**inputs, max_new_tokens=256, do_sample=True,
                                          temperature=0.7, top_p=0.8, top_k=20,
                                          pad_token_id=tokenizer.eos_token_id)
                ids = out[0][len(inputs.input_ids[0]):].tolist()
                response_text = tokenizer.decode(ids, skip_special_tokens=False)

            # Tool 호출 처리
            import re as _re
            tool_matches = _re.findall(r"<tool_call>(.*?)</tool_call>",
                                        response_text, _re.DOTALL)
            if not tool_matches: break
            messages.append({"role": "assistant", "content": response_text})
            for tc_str in tool_matches:
                try:
                    tc = json.loads(tc_str.strip())
                    fn = tc.get("name", ""); args = tc.get("arguments", {})
                    res = TOOL_FUNCS[fn](**args) if fn in TOOL_FUNCS else f"미등록: {fn}"
                    if verbose: print(f"  [Tool] {fn} -> {res}")
                    messages.append({"role": "tool", "name": fn, "content": str(res)})
                except Exception as e:
                    messages.append({"role": "tool", "name": "error", "content": str(e)})

        # 히스토리 업데이트
        clean_answer = _re.sub(r"<[^>]+>", "", response_text).strip()
        self.history.append({"role": "user",      "content": query})
        self.history.append({"role": "assistant", "content": clean_answer})
        if len(self.history) > 10:
            self.history = self.history[-10:]
        _clear_cache()
        return clean_answer

    def reset(self):
        self.history.clear(); self.turn = 0
        print("대화 기록 초기화")


bot = FinancialChatbot()
print("FinancialChatbot 준비 완료")


In [ ]:
# [실습 7-2] 멀티턴 대화 테스트

bot.reset()

turns = [
    "삼성전자 2024년 매출액과 영업이익을 알려주세요.",
    "그러면 영업이익률은 얼마인가요?",
    "전기 대비 실적 변화를 요약해주세요.",
]

for q in turns:
    bot.chat(q, use_rag=True, streaming=True, verbose=True)
    print("-" * 65)


---

## 핵심 개념 정리

| 개념 | 설명 |
|:---|:---|
| **RAG** | 외부 문서 검색 결과를 프롬프트에 포함해 환각을 줄임 |
| **임베딩** | 텍스트 -> 고차원 실수 벡터 (의미 유사도 측정 가능) |
| **FAISS** | 대용량 벡터의 고속 유사도 검색 라이브러리 |
| **청킹** | 문서를 검색 가능한 단위로 분할하는 전략 |
| **TextIteratorStreamer** | 토큰 생성 즉시 전달하는 스트리밍 인터페이스 |
| **Tool Calling** | LLM이 외부 함수를 JSON 형식으로 호출하는 기능 |
| **ReAct** | Thought -> Action -> Observation 반복 루프 |
| **Agentic AI** | 도구 + 계획 + 피드백 루프로 복잡한 목표를 달성하는 AI |

---